In [13]:
from ipag_gin.graph.ipag_builder import IPAGBuilder
from ipag_gin.graph.build_language import LanguageBuilder
from transformers import RobertaTokenizer, RobertaModel
import torch
import numpy as np
import pandas as pd
from collections import Counter
from pathlib import Path
import pickle


def process_single_code(code: str, language: str):
    """
    Process a single code sample and return its IPAG representation.
    
    Args:
        code: Source code string to process
        language: Programming language (e.g., 'c', 'cpp', 'python')
    
    Returns:
        dict: Dictionary containing 'ipag_nodes' and 'ipag_edges'
    """
    # Validate inputs
    if not code or not isinstance(code, str) or code.strip() == "":
        raise ValueError("Code must be a non-empty string")
    
    if not language or not isinstance(language, str):
        raise ValueError("Language must be a non-empty string")
    
    # Normalize language to lowercase
    language = language.lower()
    language = "cpp" if language == "c++" else language
    
    # Build language map for the single language
    lang_map = LanguageBuilder({language})
    
    # Build IPAG for single code sample
    ipag = IPAGBuilder(
        source=pd.Series([code]), 
        language=pd.Series([language]), 
        lang_map=lang_map.build()
    )
    ipag.build()
    
    # Get IPAG dataframe
    df_ipag = ipag.get_ipag_dataframe()
    
    # Extract nodes and edges for the single sample
    ipag_nodes = df_ipag['ipag_nodes'].iloc[0]
    ipag_edges = df_ipag['ipag_edges'].iloc[0]
    
    return {
        'ipag_nodes': ipag_nodes,
        'ipag_edges': ipag_edges
    }



def extract_ipag_features(ipag_nodes, ipag_edges, device=None, cuda_device_idx=0):
    """
    Extract comprehensive features from a single IPAG using GraphCodeBERT.
    
    Args:
        ipag_nodes (list): List of node dictionaries from IPAG
        ipag_edges (list): List of edge dictionaries from IPAG
        device (str or None): Torch device string; e.g., 'cuda', 'cpu'
        cuda_device_idx (int): CUDA device index for multi-GPU systems
    
    Returns:
        dict: Dictionary containing:
            - 'node_features': dict mapping node_id to feature dict
            - 'edge_features': list of edge feature dicts
            - 'graph_features': dict of graph-level features
    """
    # === Initialize Model ===
    tokenizer = RobertaTokenizer.from_pretrained("microsoft/graphcodebert-base")
    model = RobertaModel.from_pretrained("microsoft/graphcodebert-base")
    model.eval()
    
    # Device selection
    if device is None:
        if torch.cuda.is_available():
            device = torch.device(f'cuda:{cuda_device_idx}')
        else:
            device = torch.device('cpu')
    else:
        device = torch.device(device)
    
    model.to(device)
    
    # === Helper: Encode text ===
    def encode_text(text, max_length=64):
        sanitized = text if isinstance(text, str) else ""
        inputs = tokenizer(
            sanitized,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            embedding = outputs.last_hidden_state[0, 0, :].cpu().numpy()
        
        return embedding
    
    # === Helper: Get node type encoding ===
    def get_node_type_encoding(node_type):
        type_map = {'TOKEN': 0, 'DECLARATION': 1, 'PROPERTY': 2}
        encoding = np.zeros(3, dtype=np.float32)
        
        if node_type in type_map:
            encoding[type_map[node_type]] = 1.0
        else:
            encoding[2] = 1.0  # Default to PROPERTY
        
        return encoding
    
    # === Helper: Compute structural features ===
    def compute_structural_features(node_ids, edges):
        in_counts = Counter()
        out_counts = Counter()
        
        for edge in edges:
            out_counts[edge['source']] += 1
            in_counts[edge['target']] += 1
        
        features = {}
        for node_id in node_ids:
            in_degree = in_counts.get(node_id, 0)
            out_degree = out_counts.get(node_id, 0)
            
            features[node_id] = {
                'degree': in_degree + out_degree,
                'in_degree': in_degree,
                'out_degree': out_degree,
                'is_leaf': 1 if out_degree == 0 else 0,
                'is_root': 1 if in_degree == 0 else 0
            }
        
        return features
    
    # === Extract Node Features ===
    if not ipag_nodes:
        return {
            'node_features': {},
            'edge_features': [],
            'graph_features': {}
        }
    
    # Get embeddings for all nodes
    node_embeddings = {}
    for node in ipag_nodes:
        node_id = node['id']
        label = node.get('label', '')
        node_type = node.get('type', 'PROPERTY')
        original_type = node.get('original_type', '')
        
        # Create text representation
        if label:
            text = f"{node_type}: {label}"
        else:
            text = f"{node_type}: {original_type}"
        
        # Get embedding
        embedding = encode_text(text)
        node_embeddings[node_id] = embedding
    
    # Get structural features
    node_ids = [node['id'] for node in ipag_nodes]
    structural_features = compute_structural_features(node_ids, ipag_edges)
    
    # Combine all node features
    node_features = {}
    for node in ipag_nodes:
        node_id = node['id']
        node_type = node.get('type', 'PROPERTY')
        
        embedding = node_embeddings.get(node_id, np.zeros(768, dtype=np.float32))
        type_encoding = get_node_type_encoding(node_type)
        structural = structural_features.get(node_id, {
            'degree': 0, 'in_degree': 0, 'out_degree': 0,
            'is_leaf': 0, 'is_root': 0
        })
        
        structural_vector = np.array([
            structural['degree'],
            structural['in_degree'],
            structural['out_degree'],
            structural['is_leaf'],
            structural['is_root']
        ], dtype=np.float32)
        
        # Combine all features
        combined = np.concatenate([
            embedding,
            type_encoding,
            structural_vector
        ])
        
        node_features[node_id] = {
            'embedding': embedding,
            'type_encoding': type_encoding,
            'structural': structural_vector,
            'combined': combined,
            'metadata': {
                'type': node_type,
                'label': node.get('label', ''),
                'original_type': node.get('original_type', ''),
                **structural
            }
        }
    
    # === Extract Edge Features ===
    edge_features = []
    
    for edge in ipag_edges:
        source_id = edge['source']
        target_id = edge['target']
        
        source_feat = node_features.get(source_id)
        target_feat = node_features.get(target_id)
        
        if not source_feat or not target_feat:
            continue
        
        source_emb = source_feat['embedding']
        target_emb = target_feat['embedding']
        
        # Edge features
        concatenated = np.concatenate([source_emb, target_emb])
        element_wise_product = source_emb * target_emb
        
        # Cosine similarity
        norm_source = np.linalg.norm(source_emb)
        norm_target = np.linalg.norm(target_emb)
        cosine_sim = np.dot(source_emb, target_emb) / (norm_source * norm_target + 1e-8)
        
        edge_feature = {
            'source': source_id,
            'target': target_id,
            'type': edge.get('type', 'CHILD'),
            'source_embedding': source_emb,
            'target_embedding': target_emb,
            'concatenated': concatenated,
            'element_wise_product': element_wise_product,
            'cosine_similarity': cosine_sim
        }
        edge_features.append(edge_feature)
    
    # === Extract Graph-Level Features ===
    num_nodes = len(ipag_nodes)
    num_edges = len(ipag_edges)
    
    node_type_counts = Counter(node.get('type') for node in ipag_nodes)
    
    # Graph statistics
    if node_features:
        all_embeddings = np.array([feat['embedding'] for feat in node_features.values()])
        avg_embedding = np.mean(all_embeddings, axis=0)
        
        degrees = np.array([feat['metadata']['degree'] for feat in node_features.values()])
        avg_degree = np.mean(degrees)
        max_degree = np.max(degrees)
        
        num_leaves = sum(1 for feat in node_features.values() if feat['metadata']['is_leaf'])
        num_roots = sum(1 for feat in node_features.values() if feat['metadata']['is_root'])
    else:
        avg_embedding = np.zeros(768)
        avg_degree = 0
        max_degree = 0
        num_leaves = 0
        num_roots = 0
    
    graph_features = {
        'num_nodes': num_nodes,
        'num_edges': num_edges,
        'num_tokens': node_type_counts.get('TOKEN', 0),
        'num_declarations': node_type_counts.get('DECLARATION', 0),
        'num_properties': node_type_counts.get('PROPERTY', 0),
        'avg_degree': float(avg_degree),
        'max_degree': int(max_degree),
        'num_leaves': num_leaves,
        'num_roots': num_roots,
        'avg_embedding': avg_embedding,
        'graph_density': num_edges / (num_nodes * (num_nodes - 1)) if num_nodes > 1 else 0
    }
    
    return {
        'node_features': node_features,
        'edge_features': edge_features,
        'graph_features': graph_features
    }

def code_to_features_pkl(code: str, language: str, output_path: str, cuda_device_idx: int = 0):
    """
    Convert a single code sample to a features pickle file.
    
    Args:
        code (str): Source code string to process
        language (str): Programming language ('c', 'cpp', 'python', 'java', etc.)
        output_path (str): Path where the .pkl file will be saved
        cuda_device_idx (int): CUDA device index (default: 0)
    
    Returns:
        dict: The features dictionary that was saved
    """
    print(f"Processing {language} code...")
    print(f"Code length: {len(code)} characters")
    
    # Step 1: Process code to IPAG representation
    print("\n[Step 1] Building IPAG representation...")
    ipag_data = process_single_code(code, language)
    
    ipag_nodes = ipag_data['ipag_nodes']
    ipag_edges = ipag_data['ipag_edges']
    
    print(f"  - Number of nodes: {len(ipag_nodes)}")
    print(f"  - Number of edges: {len(ipag_edges)}")
    
    # Step 2: Extract features from IPAG
    print("\n[Step 2] Extracting features with GraphCodeBERT...")
    features = extract_ipag_features(
        ipag_nodes=ipag_nodes,
        ipag_edges=ipag_edges,
        device=None,  # Auto-detect (will use CUDA if available)
        cuda_device_idx=cuda_device_idx
    )
    
    print(f"  - Node features extracted: {len(features['node_features'])}")
    print(f"  - Edge features extracted: {len(features['edge_features'])}")
    print(f"  - Graph features: {len(features['graph_features'])} metrics")
    
    # Step 3: Save to pickle file
    print(f"\n[Step 3] Saving features to {output_path}...")
    
    # Create directory if it doesn't exist
    output_dir = Path(output_path).parent
    output_dir.mkdir(parents=True, exist_ok=True)
    
    with open(output_path, 'wb') as f:
        pickle.dump(features, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    # Verify file was created
    file_size = Path(output_path).stat().st_size
    print(f"  ✓ Features saved successfully ({file_size:,} bytes)")
    
    return features




In [21]:
def main():
    """Main function to convert code to features pickle file."""
    
    # ========== CONFIGURE HERE ==========
    
    # Your code to process
    code = """
def quicksort(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return quicksort(left) + middle + quicksort(right)

numbers = [3, 6, 8, 10, 1, 2, 1]
sorted_numbers = quicksort(numbers)
print(sorted_numbers)
"""
    
    # Programming language
    language = 'py'  # Options: 'py', 'c', 'cpp', 'java', etc.
    
    # Output file path
    output_path = 'code_features.pkl'
    
    # CUDA device (if you have multiple GPUs)
    cuda_device = 0
    
    # ====================================
    
    print("="*60)
    print("Code to Features Conversion")
    print("="*60)
    
    # Step 1: Build IPAG
    print("\n[1/3] Building IPAG representation...")
    ipag_data = process_single_code(code, language)
    print(f"✓ IPAG built: {len(ipag_data['ipag_nodes'])} nodes, "
          f"{len(ipag_data['ipag_edges'])} edges")
    
    # Step 2: Extract features
    print("\n[2/3] Extracting features with GraphCodeBERT...")
    features = extract_ipag_features(
        ipag_nodes=ipag_data['ipag_nodes'],
        ipag_edges=ipag_data['ipag_edges'],
        device=None,  # Auto-detect CUDA/CPU
        cuda_device_idx=cuda_device
    )
    print(f"✓ Features extracted:")
    print(f"  - {len(features['node_features'])} node features")
    print(f"  - {len(features['edge_features'])} edge features")
    print(f"  - Graph-level statistics computed")
    
    # Step 3: Save to pickle
    print(f"\n[3/3] Saving to {output_path}...")
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    
    with open(output_path, 'wb') as f:
        pickle.dump(features, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    file_size = Path(output_path).stat().st_size / 1024  # KB
    print(f"✓ Saved successfully ({file_size:.1f} KB)")
    
    # Display summary
    print("\n" + "="*60)
    print("Feature Summary:")
    print("="*60)
    gf = features['graph_features']
    print(f"Total nodes:        {gf['num_nodes']}")
    print(f"Total edges:        {gf['num_edges']}")
    print(f"Token nodes:        {gf['num_tokens']}")
    print(f"Declaration nodes:  {gf['num_declarations']}")
    print(f"Property nodes:     {gf['num_properties']}")
    print(f"Average degree:     {gf['avg_degree']:.2f}")
    print(f"Max degree:         {gf['max_degree']}")
    print(f"Graph density:      {gf['graph_density']:.4f}")
    print("="*60)
    
    # How to load the file later
    print("\nTo load the features later:")
    print(f"  with open('{output_path}', 'rb') as f:")
    print(f"      features = pickle.load(f)")
    
    return features


In [22]:
features = main()

Code to Features Conversion

[1/3] Building IPAG representation...
Building ASTs for all code snippets
Finished building ASTs
Successful: 1, Failed: 0, Total: 1
IPAG Construction Complete
Total snippets: 1
Average nodes per snippet: 156.0
Average edges per snippet: 155.0
✓ IPAG built: 156 nodes, 155 edges

[2/3] Extracting features with GraphCodeBERT...


Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Features extracted:
  - 156 node features
  - 155 edge features
  - Graph-level statistics computed

[3/3] Saving to code_features.pkl...
✓ Saved successfully (2391.1 KB)

Feature Summary:
Total nodes:        156
Total edges:        155
Token nodes:        0
Declaration nodes:  0
Property nodes:     156
Average degree:     1.99
Max degree:         16
Graph density:      0.0064

To load the features later:
  with open('code_features.pkl', 'rb') as f:
      features = pickle.load(f)
